In [13]:
# ============================================================
# Bronze Layer: Read raw JSON from Files section
# Save as Delta table — no transformation, raw data only
# ============================================================

from pyspark.sql.functions import current_timestamp, input_file_name

# Read all JSON files from the landing folder
# multiLine=True handles the array of objects format
df_raw = spark.read.option("multiLine", "true") \
              .json("Files/raw/countries/*.json")

# Add metadata columns
df_raw = df_raw \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", input_file_name())

# Save as Delta table in Bronze
df_raw.write.format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable("bronze_countries_raw")

print(f"Rows loaded: {df_raw.count()}")
print(f"Columns: {len(df_raw.columns)}")

StatementMeta(, 8a8de77c-bc24-4de0-a199-5fd0edbf3568, 15, Finished, Available, Finished, False)

Rows loaded: 12
Columns: 3


In [16]:
# ALWAYS run this before building Silver
# You need to understand every nested field

df_raw = spark.table("bronze_countries_raw")

print("=== SCHEMA ===")
df_raw.printSchema()

print("\n=== SAMPLE RECORD ===")
import json
sample = df_raw.limit(1).toPandas().to_dict(orient='records')[0]
print(json.dumps({k: str(v)[:200] for k, v in sample.items()}, indent=2))

StatementMeta(, 8a8de77c-bc24-4de0-a199-5fd0edbf3568, 18, Finished, Available, Finished, False)

=== SCHEMA ===
root
 |-- data: struct (nullable = true)
 |    |-- meta: struct (nullable = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- duration: long (nullable = true)
 |    |    |-- limit: long (nullable = true)
 |    |    |-- more: boolean (nullable = true)
 |    |    |-- offset: long (nullable = true)
 |    |    |-- request_id: string (nullable = true)
 |    |    |-- total: long (nullable = true)
 |    |-- objects: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _meta: struct (nullable = true)
 |    |    |    |    |-- lastUpdatedTimestamp: long (nullable = true)
 |    |    |    |-- area: struct (nullable = true)
 |    |    |    |    |-- kilometers: double (nullable = true)
 |    |    |    |    |-- miles: double (nullable = true)
 |    |    |    |-- assets: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |-- borders: array (nullable = true)
 |    |    |    |  

In [3]:
# ALWAYS run this before building Silver
# You need to understand every nested field

df_raw = spark.table("bronze_countries_raw")

print("=== SCHEMA ===")
df_raw.printSchema()

print("\n=== SAMPLE RECORD ===")
import json
sample = df_raw.limit(1).toPandas().to_dict(orient='records')[0]
print(json.dumps({k: str(v)[:200] for k, v in sample.items()}, indent=2))

StatementMeta(, 8c90f2da-f347-4d2f-b26c-97f5cbb78a89, 5, Finished, Available, Finished, False)

=== SCHEMA ===
root
 |-- data: struct (nullable = true)
 |    |-- meta: struct (nullable = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- duration: long (nullable = true)
 |    |    |-- limit: long (nullable = true)
 |    |    |-- more: boolean (nullable = true)
 |    |    |-- offset: long (nullable = true)
 |    |    |-- request_id: string (nullable = true)
 |    |    |-- total: long (nullable = true)
 |    |-- objects: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _meta: struct (nullable = true)
 |    |    |    |    |-- lastUpdatedTimestamp: long (nullable = true)
 |    |    |    |-- area: struct (nullable = true)
 |    |    |    |    |-- kilometers: long (nullable = true)
 |    |    |    |    |-- miles: double (nullable = true)
 |    |    |    |-- assets: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |-- borders: array (nullable = true)
 |    |    |    |    

In [18]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

df_raw = spark.table("bronze_countries_raw")

# 1) Pagination check: did each page get a different offset?
print("=== PAGINATION CHECK ===")
df_raw.select(
    col("data.meta.offset").alias("offset"),
    col("data.meta.limit").alias("limit"),
    col("data.meta.count").alias("count"),
    col("data.meta.total").alias("total"),
    col("data.meta.more").alias("more"),
).orderBy("offset").show(20, truncate=False)

# 2) Explode: one row per page -> one row per country
df_countries = (
    df_raw
    .select(F.explode("data.objects").alias("c"), "_source_file")
    .select("c.*", "_source_file")
)

print("=== ROW COUNT ===")
total = df_countries.count()
print(f"Total countries: {total}")          # expect 254
print(f"Distinct countries (by uuid): {df_countries.select('uuid').distinct().count()}")

# 3) Null checks
print("\n=== NULL CHECKS on key fields ===")
checks = {
    "region": col("region"),
    "subregion": col("subregion"),
    "population": col("population"),
    "area_km": col("area.kilometers"),
}
for name, expr in checks.items():
    print(f"{name}: {df_countries.filter(expr.isNull()).count()} nulls")

# 4) Distincts
print("\n=== DISTINCT REGIONS ===")
df_countries.select("region").distinct().orderBy("region").show(20, truncate=False)

print("\n=== DISTINCT SUBREGIONS ===")
df_countries.select("subregion").distinct().orderBy("subregion").show(30, truncate=False)

# 5) Population range
print("\n=== POPULATION RANGE ===")
df_countries.select(col("population").cast("long")).summary("min", "max", "mean").show()

StatementMeta(, 8a8de77c-bc24-4de0-a199-5fd0edbf3568, 20, Finished, Available, Finished, False)

=== PAGINATION CHECK ===
+------+-----+-----+-----+-----+
|offset|limit|count|total|more |
+------+-----+-----+-----+-----+
|0     |25   |25   |254  |true |
|25    |25   |25   |254  |true |
|50    |25   |25   |254  |true |
|75    |25   |25   |254  |true |
|100   |25   |25   |254  |true |
|125   |25   |25   |254  |true |
|150   |25   |25   |254  |true |
|175   |25   |25   |254  |true |
|200   |25   |25   |254  |true |
|225   |25   |25   |254  |true |
|250   |25   |4    |254  |false|
|275   |25   |0    |254  |false|
+------+-----+-----+-----+-----+

=== ROW COUNT ===
Total countries: 254
Distinct countries (by uuid): 254

=== NULL CHECKS on key fields ===
region: 0 nulls
subregion: 0 nulls
population: 0 nulls
area_km: 0 nulls

=== DISTINCT REGIONS ===
+---------+
|region   |
+---------+
|Africa   |
|Americas |
|Antarctic|
|Asia     |
|Europe   |
|Oceania  |
+---------+


=== DISTINCT SUBREGIONS ===
+-------------------------+
|subregion                |
+-------------------------+
|     